<a href="https://colab.research.google.com/github/sgterrado/Need4Data-Blue-Bumper-Cars/blob/Ann%2FData-clean/Need4Data_DataClean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning

**Leading question:**

How does weather and time of day affect car accidents?

In [219]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [220]:
#Removes Science Notation from df.describe() and other functions
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

In [221]:
df = pd.read_csv('/content/US_Accidents_March23_sampled_500k.csv', encoding ='latin1')

In [222]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 46 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ID                     500000 non-null  object 
 1   Source                 500000 non-null  object 
 2   Severity               500000 non-null  int64  
 3   Start_Time             500000 non-null  object 
 4   End_Time               500000 non-null  object 
 5   Start_Lat              500000 non-null  float64
 6   Start_Lng              500000 non-null  float64
 7   End_Lat                279623 non-null  float64
 8   End_Lng                279623 non-null  float64
 9   Distance(mi)           500000 non-null  float64
 10  Description            499999 non-null  object 
 11  Street                 499309 non-null  object 
 12  City                   499981 non-null  object 
 13  County                 500000 non-null  object 
 14  State                  500000 non-nu

In [223]:
#Check for duplicates
df[df.duplicated()]

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight


There is no duplicate to drop.

In [224]:
#check for n/a
df.isna().mean()*100


,0
ID,0.00
Source,0.00
Severity,0.00
Start_Time,0.00
End_Time,0.00
Start_Lat,0.00
Start_Lng,0.00
End_Lat,44.08
End_Lng,44.08
Distance(mi),0.00


Most concerning column with missing data would be `End_lat` and `End_lng` for having more than 44% of missing value. For the dataset I will drop `End_lat` and `Eng_lng` for the alarming high amount of missing values and their not being able to contribute to our team leading question "How does weather and time of day affect car accidents?" I will also drop column ID for it is just an indentifier and doesn't give useful infomation.
Base on our team leading question  **how weather and time of day affet traffic?**, I will also drop the column that will not give us useful weather infomation : `Source, Description,Street, Airport_Code, Bump, Crossing, Give_Way, Junction, Railway, Roundabout, Stop, Traffic_Calming, Traffic_Signal, Turning_Loop, Amenity`

In [225]:
#Drop columns with too many NaN values and ID
df = df.drop(columns = ['End_Lat', 'End_Lng', 'ID', 'Description', 'Street',
                        'Airport_Code', 'Bump', 'Crossing', 'Give_Way',
                        'Junction', 'No_Exit', 'Railway', 'Roundabout',
                        'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal',
                        'Turning_Loop', 'Amenity', 'Source'])

Now I need to identify potential incorrect data type and fix them.

In [226]:
df['Country'].unique()

array(['US'], dtype=object)

I will drop `Timezone` and `Country` for all the entry in the dataset is set in the US and `Start_Time` and `End_Time` are recorded in local time.

In [227]:
df = df.drop(columns= ['Timezone', 'Country'])

In [228]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 24 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Severity               500000 non-null  int64  
 1   Start_Time             500000 non-null  object 
 2   End_Time               500000 non-null  object 
 3   Start_Lat              500000 non-null  float64
 4   Start_Lng              500000 non-null  float64
 5   Distance(mi)           500000 non-null  float64
 6   City                   499981 non-null  object 
 7   County                 500000 non-null  object 
 8   State                  500000 non-null  object 
 9   Zipcode                499884 non-null  object 
 10  Weather_Timestamp      492326 non-null  object 
 11  Temperature(F)         489534 non-null  float64
 12  Wind_Chill(F)          370983 non-null  float64
 13  Humidity(%)            488870 non-null  float64
 14  Pressure(in)           491072 non-nu

Out of the two `Zipcode` and `City`

We only need 1 daytime/nightime identifier from `Sunrise_Sunset, Civil_Twilight, Astronomical_Twilight, Nautical_Twilight`




In [229]:
#find which will be the most optimal day/night identifier
df['Sunrise_Sunset'].isna().equals(df['Civil_Twilight'].isna()) and \
df['Sunrise_Sunset'].isna().equals(df['Nautical_Twilight'].isna()) and \
df['Sunrise_Sunset'].isna().equals(df['Astronomical_Twilight'].isna())



True

Since `Sunrise_Sunset, Civil_Twilight, Astronomical_Twilight, Nautical_Twilight` share the shame missing value pattern, I will keep the most understandable one which is the `Sunrise_Sunset`. And since the missing value rate in small (0.30%) I will drop those rows.

In [230]:
df = df.drop( columns= ['Civil_Twilight', 'Astronomical_Twilight',
                        'Nautical_Twilight'])

df.dropna(subset =['Sunrise_Sunset'], inplace = True)

In [231]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 498517 entries, 0 to 499999
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Severity           498517 non-null  int64  
 1   Start_Time         498517 non-null  object 
 2   End_Time           498517 non-null  object 
 3   Start_Lat          498517 non-null  float64
 4   Start_Lng          498517 non-null  float64
 5   Distance(mi)       498517 non-null  float64
 6   City               498517 non-null  object 
 7   County             498517 non-null  object 
 8   State              498517 non-null  object 
 9   Zipcode            498402 non-null  object 
 10  Weather_Timestamp  491004 non-null  object 
 11  Temperature(F)     488221 non-null  float64
 12  Wind_Chill(F)      369686 non-null  float64
 13  Humidity(%)        487561 non-null  float64
 14  Pressure(in)       489757 non-null  float64
 15  Visibility(mi)     487398 non-null  float64
 16  Wind_Di

In [232]:
df.head(5)

,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,County,State,Zipcode,...,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Sunrise_Sunset
0,2,2019-06-12 10:10:56,2019-06-12 10:55:58,30.64,-91.15,0.00,Zachary,East Baton Rouge,LA,70791-4610,...,77.00,77.00,62.00,29.92,10.00,NW,5.00,0.00,Fair,Day
1,2,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,38.99,-77.40,0.06,Sterling,Loudoun,VA,20164-2813,...,45.00,43.00,48.00,29.91,10.00,W,5.00,0.00,Fair,Night
2,2,2022-08-20 13:13:00.000000000,2022-08-20 15:22:45.000000000,34.66,-120.49,0.02,Lompoc,Santa Barbara,CA,93436,...,68.00,68.00,73.00,29.79,10.00,W,13.00,0.00,Fair,Day
3,2,2022-02-21 17:43:04,2022-02-21 19:43:23,43.68,-92.99,1.05,Austin,Mower,MN,55912,...,27.00,15.00,86.00,28.49,10.00,ENE,15.00,0.00,Wintry Mix,Day
4,2,2020-12-04 01:46:00,2020-12-04 04:13:09,35.40,-118.99,0.05,Bakersfield,Kern,CA,93305-2649,...,42.00,42.00,34.00,29.77,10.00,CALM,0.00,0.00,Fair,Night


 Found mismatch Start_Time, End_Time, Zipcode layout, fixing that first


In [233]:
df['Start_Time'] = pd.to_datetime(
    df['Start_Time'],
    errors = 'coerce',
    format = 'mixed'
)

df['End_Time'] = pd.to_datetime(
    df['End_Time'],
    errors = 'coerce',
    format = 'mixed'
)

In [234]:
df.loc[~df['Zipcode'].astype(str).str.match(r'^\d{5}(-\d{4})?$', na=False)]['Zipcode'].unique()

array(['93247-94ND', '91105-18ND', '30518-17ND', nan, '93210-97ND',
       '44039-40ND', '91214-28ND', '44105-56ND', '44104-27ND',
       '30336-15ND', '32025-26ND', '29829-99ND', '30309-28ND',
       '93314-90ND', '32034-86ND', '23230-43ND', '77002-76ND',
       '34232-62ND', '33543-91ND', '70119-39ND', '23231-70ND',
       '23185-59ND', '23231-55ND', '30143-68ND', '93240-91ND',
       '23434-76ND', '93436-61ND', '93117-15ND', '30139-28ND',
       '29588-62ND', '93455-60ND', '23223-79ND', '44310-32ND',
       '75701-92ND', '30097-15ND', '29588-68ND', '11215-62ND',
       '48120-12ND', '11214-63ND', '90808-28ND', '17372-95ND',
       '30506-30ND', '23220-10ND', '23225-42ND', '22902-71ND',
       '95110-17ND', '30354-33ND', '17551-97ND', '75204-64ND',
       '19046-14ND', '29492-78ND', '44256-94ND', '77012-33ND',
       '19147-17ND', '32505-49ND', '23226-26ND', '30230-37ND',
       '77041-61ND', '93291-29ND', '75215-33ND', '17547-15ND',
       '75703-64ND', '44333-29ND', '30540-29ND', '

In [235]:
df['Zipcode'] = df['Zipcode'].astype(str).str[:5] #stip to get 5 zip standard
df['Zipcode'].value_counts(dropna = False)

,count
Zipcode,
27610,846
32819,788
33186,745
91761,739
70808,716
...,...
05001,1
44450,1
03261,1


In [236]:
df['State'].unique()

array(['LA', 'VA', 'CA', 'MN', 'MA', 'OR', 'FL', 'TX', 'IN', 'MT', 'AL',
       'AZ', 'NY', 'NC', 'SC', 'KS', 'MI', 'CO', 'GA', 'MD', 'IL', 'PA',
       'NM', 'NV', 'NE', 'NJ', 'TN', 'UT', 'RI', 'MS', 'IA', 'WA', 'MO',
       'OH', 'OK', 'CT', 'DC', 'AR', 'WV', 'KY', 'WI', 'NH', 'WY', 'DE',
       'VT', 'ID', 'ME', 'ND', 'SD'], dtype=object)

In [237]:
df['Weather_Condition'].unique()

array(['Fair', 'Wintry Mix', 'Light Rain', 'Cloudy', 'Mostly Cloudy',
       'Partly Cloudy', 'Clear', 'Scattered Clouds', 'Fog', 'Overcast',
       'Light Snow', 'T-Storm', nan, 'Thunderstorms and Rain', 'Thunder',
       'Light Rain with Thunder', 'Rain', 'Showers in the Vicinity',
       'Mostly Cloudy / Windy', 'Heavy Rain', 'Cloudy / Windy',
       'Light Drizzle', 'Heavy T-Storm', 'Light Rain / Windy', 'Smoke',
       'Haze', 'Blowing Dust / Windy', 'N/A Precipitation',
       'Thunder in the Vicinity', 'Snow', 'Heavy Thunderstorms and Rain',
       'Shallow Fog', 'Light Freezing Drizzle', 'Fair / Windy',
       'Patches of Fog', 'Light Snow / Windy', 'Blowing Snow / Windy',
       'Thunderstorm', 'Drizzle', 'T-Storm / Windy',
       'Partly Cloudy / Windy', 'Heavy Rain / Windy',
       'Heavy Snow / Windy', 'Mist', 'Light Thunderstorms and Rain',
       'Rain / Windy', 'Light Freezing Rain', 'Heavy Snow',
       'Light Ice Pellets', 'Heavy T-Storm / Windy', 'Heavy Drizzle',
    

In [238]:
df['Wind_Direction'].unique()

array(['NW', 'W', 'ENE', 'CALM', 'SW', 'VAR', 'S', 'E', 'WSW', 'NNE',
       'Variable', 'West', 'N', 'ESE', 'NNW', 'North', 'SSW', 'WNW', 'NE',
       'Calm', 'SE', 'East', 'SSE', 'South', nan], dtype=object)

In [239]:
df['Wind_Direction'] = df['Wind_Direction'].replace({
    "North" : "N",
    "East" : "E",
    "South" : "S",
    "Calm" : "CALM",
    "West" : "W",
    "Variable" : "VAR"
})

In [240]:
df['Wind_Direction'].value_counts(dropna = False)

,count
Wind_Direction,
CALM,86076
S,38442
W,35591
N,29711
SSW,24723
E,24476
WNW,24281
NW,23836
SW,23592
